# Aula 04 — Ciclo de vida do grafo e acúmulo de gradientes

Laboratório diagnóstico: separar valores, grafo e buffers de derivadas. Execute em ordem, em kernel novo. Os erros deliberados são capturados e conferidos; nenhum erro deve interromper a execução.

[Aula](../aulas/04-grafo-acumulo-gradientes.md) · [Currículo](../README.md)

## Objetivos e método

Conferir folhas, acúmulo, retenção, desconexão, mutação e equivalência entre lote completo e micro-lotes. Usamos fixtures sintéticas em CPU/float64, sem treino nem avaliação preditiva: não há seleção de hiperparâmetros ou teste reservado. Cada exemplo isola um contrato, e não mede generalização.

## Preparação

Mínimos: Python 3.10, PyTorch 2.6. Para abrir localmente, Jupyter e nbformat 5.10 são suficientes; para execução automatizada, nbclient 0.10. Se necessário, instale PyTorch compatível com sua plataforma antes de executar. O laboratório não baixa dados nem usa GPU. Seed: `20260904`.

In [ ]:
import platform
import torch

SEED = 20260904
DTYPE = torch.float64
generator = torch.Generator(device="cpu").manual_seed(SEED)
checks = []

def check(name, condition):
    assert bool(condition), name
    checks.append(name)

def close(name, actual, expected, atol=1e-12):
    torch.testing.assert_close(actual, torch.as_tensor(expected, dtype=actual.dtype),
                               atol=atol, rtol=1e-12)
    checks.append(name)

def rejects(name, action, fragment):
    try:
        action()
    except RuntimeError as error:
        check(name, fragment.lower() in str(error).lower())
    else:
        raise AssertionError(f"{name}: o erro esperado não ocorreu")

def scalar(value):
    return torch.tensor(value, dtype=DTYPE, requires_grad=True)

print("Python", platform.python_version(), "PyTorch", torch.__version__)
check("seed registrada", generator.initial_seed() == SEED)

## 1. Folhas e intermediários

Para $z=3x$ e $L=z^2$, em $x=2$, esperamos $dL/dz=12$ e $dL/dx=36$. `retain_grad` solicita o buffer do intermediário, sem convertê-lo em folha. Evitamos ler `.grad` de não folhas sem retenção, pois essa consulta gera um aviso.

In [ ]:
x = scalar(2.0)
z = 3 * x
check("x folha", x.is_leaf and x.grad_fn is None)
check("z intermediário", not z.is_leaf and z.grad_fn is not None)
z.retain_grad()
z.square().backward()
close("gradiente folha", x.grad, 36.0)
close("gradiente intermediário retido", z.grad, 12.0)
check("retain_grad não transforma em folha", not z.is_leaf)
print("x.grad =", x.grad.item(), "z.grad =", z.grad.item())

## 2. Acúmulo entre forwards novos

A mesma folha pode participar de grafos novos. Cada `backward` adiciona contribuições ao buffer existente. Em $x=2$, dois forwards de $x^2$ produzem 4 e depois 8 no buffer, sem alterar $x$.

In [ ]:
x = scalar(2.0)
x.square().backward()
close("primeira contribuição", x.grad, 4.0)
x.square().backward()  # novo forward, mesma folha
close("acúmulo entre grafos", x.grad, 8.0)
close("backward não atualiza peso", x.detach(), 2.0)
x.grad = None
check("buffer ausente após limpeza", x.grad is None)
x.square().backward()
close("novo acúmulo após None", x.grad, 4.0)
x.grad.zero_()
close("buffer zerado", x.grad, 0.0)
x.square().backward()
close("novo acúmulo após zero", x.grad, 4.0)
print("Sem limpeza: 8; após limpeza e novo backward:", x.grad.item())

## 3. Reusar uma loss pode falhar

O quadrado salva a entrada para o backward. Depois da primeira consulta, seus dados salvos são liberados por padrão. Limpar `.grad` não os recria. Uma soma simples pode permitir repetição sem dados salvos; isso não é um contrato geral para reusar qualquer grafo.

In [ ]:
x = scalar(2.0)
loss = x.square()
loss.backward()
x.grad = None
rejects("grafo consumido", lambda: loss.backward(), "second time")
check("grad_fn pode permanecer após backward", loss.grad_fn is not None)
x.square().backward()  # reconstrução válida
close("forward reconstruído", x.grad, 4.0)
print("Grafo consumido detectado; novo forward aprovado.")

## 4. Duas consultas deliberadas ao mesmo grafo

Se $u=x^2$, $L_1=u$ e $L_2=3u$, as contribuições são 4 e 12 em $x=2$. Reter na primeira consulta permite a segunda. A soma das losses num forward novo deve dar o mesmo 16. Não altere os parâmetros entre essas consultas.

In [ ]:
x = scalar(2.0)
u = x.square()
u.backward(retain_graph=True)
(3 * u).backward()
close("dois ramos acumulados", x.grad, 16.0)
x.grad = None
u = x.square()
(u + 3 * u).backward()
close("soma dos objetivos", x.grad, 16.0)
print("Gradiente da soma:", x.grad.item())

## 5. Reter grafo não é diferenciar o gradiente

`create_graph=True` registra as operações da derivada, permitindo derivar novamente. Para $L=x^3$, em $x=2$, tanto $L'$ quanto $L''$ valem 12. Usamos `autograd.grad` para obter retornos sem preencher `.grad`.

In [ ]:
x = scalar(2.0)
first, = torch.autograd.grad(x ** 3, x, create_graph=True)
second, = torch.autograd.grad(first, x)
close("primeira derivada", first.detach(), 12.0)
close("segunda derivada", second, 12.0)
check("grad API não preenche buffer", x.grad is None)
x = scalar(2.0)
ordinary, = torch.autograd.grad(x ** 3, x, retain_graph=True)
check("retain_graph não cria grafo da derivada", not ordinary.requires_grad)
rejects("derivada sem grafo", lambda: torch.autograd.grad(ordinary, x),
        "does not require grad")
print("Primeira e segunda derivadas:", first.item(), second.item())

## 6. `clone`, `detach` e snapshot

`clone` copia armazenamento preservando conexão; `detach` corta conexão compartilhando armazenamento; `detach().clone()` faz cópia desconectada. Os ponteiros são comparados apenas entre tensores vivos e não vazios desta fixture CPU.

In [ ]:
x = torch.tensor([1.0, 2.0], dtype=DTYPE, requires_grad=True)
copied = x.clone()
detached = x.detach()
snapshot = x.detach().clone()
check("clone independente e conectado", copied.data_ptr() != x.data_ptr()
      and copied.grad_fn is not None)
check("detach compartilha armazenamento", detached.data_ptr() == x.data_ptr()
      and not detached.requires_grad)
check("snapshot independente", snapshot.data_ptr() != x.data_ptr()
      and not snapshot.requires_grad)
copied.sum().backward()
close("clone preserva derivada", x.grad, [1.0, 1.0])
detached.add_(10)  # demonstração isolada, após consumir o grafo
close("mutação pelo alias", x.detach(), [11.0, 12.0])
close("snapshot preservado", snapshot, [1.0, 2.0])
print("Original após alias:", x.detach().tolist(), "snapshot:", snapshot.tolist())

## 7. Reativar derivadas não reconecta o passado

Desconectar $y=x^2$ e criar uma nova folha a partir desse valor permite derivar em relação a essa folha. Não recupera o caminho até $x$. Guardar `loss.item()` é apropriado para logs, mas reconstruir um tensor a partir desse número também perde o histórico.

In [ ]:
x = scalar(2.0)
y = x.square()
fresh = y.detach().clone().requires_grad_()
(3 * fresh).backward()
close("derivada na nova folha", fresh.grad, 3.0)
check("x desconectado", x.grad is None)
number = y.item()
rewrapped = scalar(number)
rewrapped.backward()
close("derivada apenas no tensor refeito", rewrapped.grad, 1.0)
check("item não reconecta x", x.grad is None)
check("log escalar Python", isinstance(number, float))
print("nova folha.grad =", fresh.grad.item(), "x.grad =", x.grad)

## 8. `no_grad` e atualização após backward

O bloco não apaga o `requires_grad` da folha nem seu buffer. Desliga o registro das operações ali executadas. Uma atualização após o backward pode ocorrer dentro desse bloco; o próximo forward volta a registrar operações.

In [ ]:
x = scalar(2.0)
x.square().backward()
with torch.no_grad():
    prediction = x.square()
    x.add_(x.grad, alpha=-0.1)
check("previsão sem grafo", not prediction.requires_grad)
check("folha ainda diferenciável", x.requires_grad and x.is_leaf)
close("peso após atualização", x.detach(), 1.6)
close("buffer antigo ainda existe", x.grad, 4.0)
x.grad = None
x.square().backward()
close("derivada no novo ponto", x.grad, 3.2)
print("Novo peso:", x.item(), "novo gradiente:", x.grad.item())

## 9. In-place: duas rejeições úteis

Primeiro tentamos alterar uma folha diferenciável em modo normal. Depois alteramos, dentro de `no_grad`, uma entrada salva por um backward ainda pendente. O segundo caso mostra que `no_grad` não autoriza destruir os valores necessários à regra da cadeia.

In [ ]:
x = scalar(2.0)
rejects("mutação direta da folha", lambda: x.add_(1), "leaf")
x = scalar(2.0)
loss = x.square()
with torch.no_grad():
    x.add_(1)
rejects("versão salva alterada", lambda: loss.backward(), "modified by an inplace")
x = scalar(2.0)
loss = x.square()
x.detach().add_(1)
rejects("alias detach também invalida versão", lambda: loss.backward(),
        "modified by an inplace")
print("Três mutações indevidas detectadas.")

## 10. Dados e referência de uma MLP

Criamos 11 exemplos com 3 entradas e 2 alvos contínuos, uma camada oculta tanh com 4 unidades e pesos fixos. A loss é metade da média dos 22 resíduos quadráticos. A função abaixo não mistura exemplos, condição necessária à equivalência por micro-lotes.

In [ ]:
X = torch.randn(11, 3, dtype=DTYPE, generator=generator)
target = torch.randn(11, 2, dtype=DTYPE, generator=generator)
initial = [torch.randn(3, 4, dtype=DTYPE, generator=generator) * 0.2,
           torch.zeros(4, dtype=DTYPE),
           torch.randn(4, 2, dtype=DTYPE, generator=generator) * 0.2,
           torch.zeros(2, dtype=DTYPE)]

def parameters():
    return [value.clone().requires_grad_() for value in initial]

def objective(inputs, targets, params):
    w1, b1, w2, b2 = params
    output = torch.tanh(inputs @ w1 + b1) @ w2 + b2
    assert output.shape == targets.shape
    return (output - targets).square().mean() / 2

full = parameters()
full_loss = objective(X, target, full)
full_loss.backward()
reference = [p.grad.clone() for p in full]
check("referência finita", all(torch.isfinite(g).all() for g in reference))
check("gradientes com shapes dos parâmetros",
      all(p.shape == g.shape for p, g in zip(full, reference)))
print("Loss integral:", full_loss.item())

## 11. Micro-lotes desiguais: 4 + 4 + 3

Cada loss local é uma média. Multiplique por $b_k/N$ antes do backward, com $N=11$. Os parâmetros ficam fixos durante todas as contribuições e cada forward tem seu próprio grafo. Não precisamos de `retain_graph=True`.

In [ ]:
micro = parameters()
ranges = [(0, 4), (4, 8), (8, 11)]
weighted_loss = 0.0
for start, stop in ranges:
    local = objective(X[start:stop], target[start:stop], micro)
    weight = (stop - start) / len(X)
    (local * weight).backward()
    weighted_loss += local.item() * weight

micro_error = max((p.grad - g).abs().max().item()
                  for p, g in zip(micro, reference))
for i, (p, g) in enumerate(zip(micro, reference)):
    close(f"micro-lotes parâmetro {i}", p.grad, g)
close("loss agregada", torch.tensor(weighted_loss, dtype=DTYPE), full_loss.detach())
check("pesos não mudaram entre lotes", all(torch.equal(p.detach(), q)
      for p, q in zip(micro, initial)))
print("Erro máximo dos gradientes:", micro_error)

## 12. Contraprova: média de três médias

Dividir cada loss por três atribui ao último lote a mesma massa dos lotes maiores. O código roda, mas deriva outro objetivo. Esta contraprova deve divergir da referência, e não ser forçada a coincidir.

In [ ]:
wrong = parameters()
for start, stop in ranges:
    (objective(X[start:stop], target[start:stop], wrong) / len(ranges)).backward()
wrong_error = max((p.grad - g).abs().max().item()
                  for p, g in zip(wrong, reference))
check("média ingênua detectada", wrong_error > 1e-3)
print("Erro da média ingênua:", wrong_error)

## 13. Zero matemático versus ausência de caminho

`None` e zero têm significados diferentes. Solicitamos explicitamente `allow_unused=True` e `materialize_grads=False` para representar uma entrada não usada por `None`. Uma entrada usada em $0x$ tem gradiente tensorial zero.

In [ ]:
x, unused = scalar(2.0), scalar(7.0)
zero, absent = torch.autograd.grad(0 * x, (x, unused), allow_unused=True,
                                  materialize_grads=False)
close("zero com caminho", zero, 0.0)
check("ausência de caminho", absent is None)
check("consultas não acumulam", x.grad is None and unused.grad is None)
print("Gradiente usado:", zero.item(), "não usado:", absent)

## Verificações e interpretação

O resumo abaixo só é emitido após todas as verificações anteriores. As fixtures demonstram contratos locais, não consumo de memória em produção nem desempenho de um modelo. A equivalência por micro-lotes pressupõe parâmetros fixos, loss separável e mesmas operações por exemplo; BatchNorm em treino e aleatoriedade não controlada quebram essa comparação simples.

In [ ]:
check("nomes de contratos únicos", len(checks) == len(set(checks)))
print(f"{len(checks)} contratos aprovados")
print({"loss": full_loss.item(), "micro_error": micro_error,
       "wrong_error": wrong_error, "seed": SEED})

## Exercícios e próximos passos

1. Refaça a partição como 5 + 6. **Resposta:** ponderar por 5/11 e 6/11 preserva a referência, dentro da tolerância.
2. Limpe os gradientes dentro de cada micro-lote. **Resposta:** apenas a última contribuição sobrevive; a equivalência é perdida.
3. Troque `detach().clone()` por `clone()`. **Resposta:** a cópia ainda tem conexão com a entrada; independência de armazenamento não implica independência do grafo.
4. Use apenas `retain_graph=True` para derivar duas vezes $x^3$. **Resposta:** reter os dados do forward não registra as operações da primeira derivada; para isso usamos `create_graph=True`.

Na Aula 05 faremos gradient checking e paridade sistemática com NumPy. O laço de treinamento completo fica na Aula 11.

### Fontes oficiais

Consultadas em 9 de setembro de 2026. APIs exercitadas em PyTorch 2.6.0+cpu; páginas 2.14 são identificadas como documentação consultada, não como ambiente executado.

- [Autograd mechanics — 2.6](https://docs.pytorch.org/docs/2.6/notes/autograd.html)
- [is_leaf — 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.is_leaf.html)
- [backward — 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.autograd.backward.html)
- [detach — 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.detach.html)
- [grad — 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.autograd.grad.html)